# PRACTICE 2 - Pretrained neural networks

Notebook lấy code, báo cáo và kết quả từ nhánh `main` của repo `kakal1090/lab2-practice2-final`. Không cần upload ZIP.

Điều kiện thực nghiệm: CIFAR-10 đầy đủ, ảnh 96×96, seed 42, 5 epoch mỗi cấu hình. Kết quả có sẵn được chạy bằng CPU; chạy notebook để xem lại không phải huấn luyện mới trên Colab.

Chạy phần 1–4 để xem kết quả, kiến trúc và TensorBoard. Muốn train lại, chọn GPU và bật `RUN_NEW_EXPERIMENTS=True` ở phần 5; dữ liệu/weights được tải khi cần. Không trộn số liệu 224×224 cũ vào bảng so sánh này.

## 1. Lấy bài từ GitHub

Chạy ô dưới trên Google Colab để lấy bài từ nhánh main. Mỗi lần chạy tạo thư mục mới và in commit để biết phiên bản đang sử dụng. Không cần upload ZIP.

In [ ]:
from google.colab import files
from pathlib import Path
import os, sys, subprocess, json, csv
import shutil, datetime, zipfile, tempfile

REPO = "https://github.com/kakal1090/lab2-practice2-final.git"
workspace = Path(tempfile.mkdtemp(prefix="lab2_github_", dir="/content"))
project = workspace / "lab2-practice2-final"
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", "main", REPO, str(project)],
    check=True
)
assert (project / "src/experiment.py").exists(), "Thiếu code"
assert (project / "results/comparison.csv").exists(), "Thiếu kết quả"
print("Đã tải bài từ GitHub:", project)
print("Commit:", subprocess.check_output(
    ["git", "-C", str(project), "rev-parse", "--short", "HEAD"], text=True
).strip())
print((project / "results/environment.json").read_text())
print("Chạy tiếp phần 2–4 để xem kết quả; chưa huấn luyện lại.")


## 2. Xem bảng thực nghiệm và báo cáo
Các số liệu đọc trực tiếp từ CSV/JSON đã chạy, không nhập tay. Báo cáo HTML có toàn bộ hình và giải thích.

In [ ]:
from IPython.display import display, HTML
with (project/'results/comparison.csv').open() as f:
    rows=list(csv.DictReader(f))
for row in rows:
    print(row['Run'], '| val=', row['Val acc (%)'], '% | test=', row['Test acc (%)'] or 'không đánh giá', '| best epoch=', row['Best epoch'])
display(HTML((project/'report/BAO_CAO_PRACTICE_2.html').read_text()))


## 3. Xem kiến trúc, các lớp và số tham số
Mỗi run lưu architecture.txt, parameters.json và config.json. So sánh head-only với layer4+fc; classifier từ ImageNet được thay thành 10 đầu ra.

In [ ]:
for run in ['resnet_baseline','densenet_baseline','resnet_finetune']:
    folder=project/'results'/run
    params=json.loads((folder/'parameters.json').read_text())
    print(run,'total=',params['total'],'trainable=',params['trainable'])
    print('Trainable layer names:',params['trainable_names'])
for run in ['resnet_baseline','densenet_baseline']:
    print('KIẾN TRÚC:',run)
    print((project/'results'/run/'architecture.txt').read_text())


## 4. Mở TensorBoard thật
Event logs của từng lượt nằm trong thư mục results đã lấy từ GitHub. Xem train_loss, val_loss, train_accuracy và val_accuracy. Đặt smoothing 0 và xem đủ 5 epoch khi chụp ảnh. Đồ thị báo cáo được vẽ từ cùng CSV, không thay thế event logs.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','tensorboard'],check=True)
get_ipython().run_line_magic('load_ext','tensorboard')
get_ipython().run_line_magic('tensorboard',f'--logdir "{project / "results"}"')


## 5. Tái lập toàn bộ — tùy chọn, cần thời gian tính toán

Nếu chỉ đọc/nộp bài đã chạy thì bỏ qua phần này. Muốn kiểm chứng, chọn GPU T4 và đổi RUN_NEW_EXPERIMENTS=True. Không đổi 96×96/seed/split khi so sánh với số liệu báo cáo. Các lượt head cache feature backbone cố định; fine-tuning chạy trên ảnh. Notebook tạo thư mục mới để không ghi đè kết quả đã cung cấp.

Thứ tự tự động: kiểm tra code → tải và xác minh dữ liệu → 5 lượt train/validation → khóa lựa chọn bằng validation → test baseline và cấu hình thắng → tạo báo cáo mới. Không chỉnh lại hyperparameter dựa trên test.

In [ ]:
RUN_NEW_EXPERIMENTS = False
if RUN_NEW_EXPERIMENTS:
    new_project=Path('/content')/('practice2_reproduce_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
    new_project.mkdir()
    for name in ['src','tests']:
        shutil.copytree(project/name,new_project/name)
    shutil.copy2(project/'requirements.txt',new_project/'requirements.txt')
    bundled=project/'data/cifar-10-python.tar.gz'
    if bundled.exists():
        (new_project/'data').mkdir()
        shutil.copy2(bundled,new_project/'data/cifar-10-python.tar.gz')
        print('Đã dùng archive CIFAR-10 có sẵn cục bộ; không tải lại qua mạng.')
    subprocess.run([sys.executable,'-m','pip','install','-r',str(new_project/'requirements.txt')],check=True)
    print('Kết quả mới sẽ lưu tại:',new_project)
else:
    print('Chế độ xem kết quả. Không train lại.')


### Tùy chọn: upload dataset đã có trên máy

Repo không chứa dataset. Giữ `UPLOAD_CIFAR=False` để tải tự động khi train lại. Nếu đã có archive CIFAR-10 Python đầy đủ, đổi thành True để upload và kiểm tra checksum. Chế độ xem kết quả không cần dataset.

In [ ]:
UPLOAD_CIFAR = False
if RUN_NEW_EXPERIMENTS and UPLOAD_CIFAR:
    import hashlib
    from torchvision.datasets import CIFAR10
    picked=files.upload()
    assert len(picked)==1
    source=Path(next(iter(picked)))
    with source.open('rb') as f:
        assert hashlib.file_digest(f,'md5').hexdigest()==CIFAR10.tgz_md5, 'File CIFAR chưa đầy đủ/không đúng Python version'
    (new_project/'data').mkdir(exist_ok=True)
    shutil.copy2(source,new_project/'data'/CIFAR10.filename)
    print('Đã nhận archive hợp lệ')


In [ ]:
def live(command,cwd):
    import signal
    process=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    try:
        for line in process.stdout:
            print(line,end='',flush=True)
        if process.wait():
            raise RuntimeError('Lệnh chưa hoàn thành; đọc lỗi phía trên. Không dùng số liệu dang dở.')
    except BaseException:
        if process.poll() is None:
            os.killpg(process.pid,signal.SIGTERM)
            try:process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid,signal.SIGKILL);process.wait()
        raise
    finally:process.stdout.close()
if RUN_NEW_EXPERIMENTS:
    live([sys.executable,'-m','unittest','discover','-s','tests','-v'],new_project)
    live([sys.executable,'-u','-m','src.download_data'],new_project)
    live([sys.executable,'-u','-m','src.experiment','--image-size','96','--epochs','5','--threads','4','--workers','0'],new_project)
    live([sys.executable,'-m','src.verify_results'],new_project)
    live([sys.executable,'-m','src.make_report'],new_project)
    live([sys.executable,'-m','src.make_pdf'],new_project)
    print('ĐÃ HOÀN THÀNH LƯỢT TÁI LẬP')


## 6. Tải kết quả tái lập nếu đã chạy
Không đóng runtime trước khi tải về. Gói dưới giữ code, log, bảng, report và event logs; không gồm dataset/cache/checkpoints lớn. Muốn giữ checkpoint, tải results/<run>/best.pt riêng.

In [ ]:
if RUN_NEW_EXPERIMENTS:
    output=Path('/content/practice2_reproduction_results.zip')
    with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as z:
        for name in ['src','tests','results','report']:
            for f in (new_project/name).rglob('*'):
                if f.is_file() and '__pycache__' not in f.parts and f.suffix!='.pt':
                    z.write(f,str(Path('practice2_complete')/f.relative_to(new_project)))
    files.download(str(output))
else:
    print('Các kết quả đã chạy nằm trong thư mục results lấy từ GitHub.')


## 7. Trình bày và nộp
Đọc phần đối chiếu yêu cầu trong báo cáo; giải thích được pretrained, freeze, fine-tuning, LR/batch và train/validation/test. Điền thông tin nhóm nếu giảng viên yêu cầu. Dùng Git nhánh riêng/PR để đưa bài mới vào repo; không ghi đè hoặc trộn với kết quả 224×224 cũ. Giữ file số liệu gốc và nguồn tham khảo.